In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib

In [3]:
df = pd.read_csv("C:\Predictive Maintainance\predictive_maintenance_dataset.csv")

df.head()

,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9
0,1/1/2015,S1F01085,0,215630672,55,0,52,6,407438,0,0,7
1,1/1/2015,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0
2,1/1/2015,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0
3,1/1/2015,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0
4,1/1/2015,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3


In [4]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124494 entries, 0 to 124493
Data columns (total 12 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   date     124494 non-null  object
 1   device   124494 non-null  object
 2   failure  124494 non-null  int64 
 3   metric1  124494 non-null  int64 
 4   metric2  124494 non-null  int64 
 5   metric3  124494 non-null  int64 
 6   metric4  124494 non-null  int64 
 7   metric5  124494 non-null  int64 
 8   metric6  124494 non-null  int64 
 9   metric7  124494 non-null  int64 
 10  metric8  124494 non-null  int64 
 11  metric9  124494 non-null  int64 
dtypes: int64(10), object(2)
memory usage: 11.4+ MB


date       0
device     0
failure    0
metric1    0
metric2    0
metric3    0
metric4    0
metric5    0
metric6    0
metric7    0
metric8    0
metric9    0
dtype: int64

In [5]:
# df['date'] = pd.to_datetime(df['date'])

# df['year'] = df['date'].dt.year
# df['month'] = df['date'].dt.month
# df['day'] = df['date'].dt.day
# df['hour'] = df['date'].dt.hour

df = df.drop('date', axis=1)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['device'] = le.fit_transform(df['device'])

In [6]:
df.head()

,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9
0,0,0,215630672,55,0,52,6,407438,0,0,7
1,2,0,61370680,0,3,0,6,403174,0,0,0
2,3,0,173295968,0,0,0,12,237394,0,0,0
3,4,0,79694024,0,0,0,6,410186,0,0,0
4,5,0,135970480,0,0,0,15,313173,0,0,3


In [7]:
x = df.drop('failure', axis=1)
y = df['failure']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [2]:
!pip install xgboost==1.7.6 imbalanced-learn --upgrade


  Attempting uninstall: sklearn-compat

    Found existing installation: sklearn-compat 0.1.3

    Uninstalling sklearn-compat-0.1.3:

      Successfully uninstalled sklearn-compat-0.1.3

  Attempting uninstall: imbalanced-learn

    Found existing installation: imbalanced-learn 0.14.0

   -------------------- ------------------- 1/2 [imbalanced-learn]
    Uninstalling imbalanced-learn-0.14.0:
   -------------------- ------------------- 1/2 [imbalanced-learn]
      Successfully uninstalled imbalanced-learn-0.14.0
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]



In [9]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

In [10]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

model.fit(x_train_resampled, y_train_resampled)

,objective,'binary:logistic'
,use_label_encoder,None
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [11]:
y_pred_prob = model.predict_proba(x_test)[:,1]

y_pred = (y_pred_prob > 0.12).astype(int)

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9826499056186996
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     24881
           1       0.01      0.28      0.02        18

    accuracy                           0.98     24899
   macro avg       0.51      0.63      0.51     24899
weighted avg       1.00      0.98      0.99     24899



In [15]:
import boto3

s3 = boto3.client(
    "s3",
    region_name="ap-south-1",
    aws_access_key_id="",      # paste here
    aws_secret_access_key=""  # paste here
)

s3.upload_file("model.tar.gz", "pred-maintain-model-rodix", "model/model.tar.gz")
print("Uploaded!")

Uploaded!
